# Pipeline Mini-4K → dataset de México → fine-tuning SAM-TP

Notebook orquestador. **No reimplementa nada**: llama a los scripts que ya están
en `ML_model/scripts/pipe_videos_online/`.

Guardalo en `ML_model/notebooks/` (o donde quieras: la celda 0 encuentra la raíz sola).

Orden: 0 setup → 1 metadata → 2 explorar → 3 select → 4 fetch → 5 clean → 6 curate → 7 chequeo.
Corré una celda a la vez y mirá la salida antes de seguir.


## 0 — Setup de rutas

El notebook vive en `ML_model/notebooks/`, asi que la raiz es `..`.
Todas las rutas salen de ahi. No cambia el directorio de trabajo.


In [22]:
import os, sys, json, subprocess
from pathlib import Path

In [23]:


# Este notebook vive en ML_model/notebooks/  ->  la raiz es la carpeta de arriba.
ML_ROOT   = Path.cwd().parent
PIPE_DIR  = ML_ROOT / "scripts" / "pipe_videos_online"
RIDES_DIR = ML_ROOT / "scripts" / "pipe_nuestras_rides"
DATA      = ML_ROOT / "data"
META_DIR  = ML_ROOT / "frodobots_metadata"      # gitignored
WORK      = DATA / "mexico"                     # todo lo de esta corrida vive aca
WORK.mkdir(parents=True, exist_ok=True)

# archivos que produce el pipeline, todos bajo WORK
SELECTED  = WORK / "selected_mexico.csv"
RAW_DIR   = WORK / "raw"
CLEAN_DIR = WORK / "clean"
INDEX     = WORK / "cleaned_index.csv"
CURATED   = WORK / "curated_mexico.csv"

sys.path.insert(0, str(PIPE_DIR))   # para importar Tool_2_Revisar_Parquet

assert PIPE_DIR.is_dir(), f"No existe {PIPE_DIR} -- corre el notebook desde ML_model/notebooks/"

for k, v in dict(ML_ROOT=ML_ROOT, PIPE_DIR=PIPE_DIR, META_DIR=META_DIR, WORK=WORK).items():
    print(f"{k:9s} = {v}   {'OK' if Path(v).exists() else '(no existe todavia)'}")


ML_ROOT   = /home/pablolube/IROS26-LaRovernetta/ML_model   OK
PIPE_DIR  = /home/pablolube/IROS26-LaRovernetta/ML_model/scripts/pipe_videos_online   OK
META_DIR  = /home/pablolube/IROS26-LaRovernetta/ML_model/frodobots_metadata   OK
WORK      = /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico   OK


In [24]:
def run(script: str, *args, cwd: Path = None) -> int:
    """Corre un script del repo mostrando la salida en vivo."""
    cwd = cwd or PIPE_DIR
    cmd = [sys.executable, script, *[str(a) for a in args]]
    print("$", " ".join(cmd), f"   (cwd={cwd})\n")
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n[exit {p.returncode}]")
    return p.returncode


## 1 — Metadata del Mini-4K

Primero **listamos** los archivos del repo (celda 1a), despues **bajamos**
solo los livianos (celda 1b). Nada de video.

> Los patrones de `allow_patterns` van **sin** `**/`: en fnmatch el `*` ya
> cruza las barras, y `**/*.parquet` exige una barra literal, por lo que se
> saltea `metadata.parquet` si esta en la raiz del repo.


In [25]:
# 1a — Ver que hay realmente en el repo (no baja nada, solo lista nombres)
from huggingface_hub import list_repo_files, snapshot_download

files = list_repo_files("BitRobot/FrodoBots-Mini-4K", repo_type="dataset")
print(f"{len(files)} archivos en el repo\n")

livianos = [f for f in files if f.endswith((".parquet", ".json", ".csv", ".md", ".txt"))]
print(f"{len(livianos)} archivos livianos (parquet/json/csv/md):")
for f in livianos[:60]:
    print("  ", f)
if len(livianos) > 60:
    print(f"   ... y {len(livianos)-60} mas")

# los parquets, que es lo que nos importa
print("\nParquets:")
for f in [f for f in files if f.endswith(".parquet")][:30]:
    print("  ", f)


/home/pablolube/IROS26-LaRovernetta/ML_model/.venv-data/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1144 archivos en el repo

2 archivos livianos (parquet/json/csv/md):
   README.md
   metadata.parquet

Parquets:
   metadata.parquet


In [26]:
# 1b — Bajar los archivos livianos
#
# OJO con los patrones: huggingface_hub filtra con fnmatch, donde "*" YA cruza
# las barras. Por eso "**/*.parquet" exige una barra literal y NO matchea un
# archivo en la raiz como "metadata.parquet". Los patrones van sin "**/".
#
# Mira la salida de la celda 1a y ajusta ALLOW si hace falta:
#   - si hay UN solo parquet de metadata en la raiz -> ["metadata.parquet"]
#   - si hay varios y los queres todos             -> ["*.parquet", "*.json", "*.csv"]

ALLOW = ["*.parquet", "*.json", "*.csv"]     # <-- EDITAR segun la celda 1a

if not any(META_DIR.rglob("*.parquet")):
    snapshot_download(
        repo_id="BitRobot/FrodoBots-Mini-4K",
        repo_type="dataset",
        allow_patterns=ALLOW,
        local_dir=str(META_DIR),
    )
else:
    print("Ya hay parquets en", META_DIR, "-- borra la carpeta si queres rebajarlos")

parquets = sorted(META_DIR.rglob("*.parquet"))
print(f"\n{len(parquets)} parquets locales:")
for p in parquets[:20]:
    print("  ", p.relative_to(META_DIR), f"{p.stat().st_size/1e6:.1f} MB")

if not parquets:
    raise SystemExit(
        "No se bajo ningun parquet. Revisa ALLOW contra los nombres reales "
        "que imprimio la celda 1a. Si el dataset es gated, corre "
        "'huggingface-cli login' primero."
    )

META_PARQUET = next((p for p in parquets if p.name == "metadata.parquet"), parquets[0])
print("\nMETA_PARQUET =", META_PARQUET)


Ya hay parquets en /home/pablolube/IROS26-LaRovernetta/ML_model/frodobots_metadata -- borra la carpeta si queres rebajarlos

1 parquets locales:
   metadata.parquet 0.7 MB

META_PARQUET = /home/pablolube/IROS26-LaRovernetta/ML_model/frodobots_metadata/metadata.parquet


## 2 — Explorar el metadata y encontrar el nombre exacto de México

Usa tu propio `Tool_2_Revisar_Parquet.explorar_parquet`. El filtro del `select`
es por coincidencia exacta, así que acá confirmás si dice `Mexico`, `México` o `MX`.

In [27]:
from Tool_2_Revisar_Parquet import explorar_parquet

df_meta = explorar_parquet(META_PARQUET)

print("\n--- paises ---")
print(df_meta["country"].value_counts().to_string())


Cargando archivo parquet desde: /home/pablolube/IROS26-LaRovernetta/ML_model/frodobots_metadata/metadata.parquet...

Dimensiones del dataset (Filas, Columnas): (14141, 27)

Primeras columnas disponibles:
['folder', 'ride_id', 'device_ref_id', 'start_utc', 'source', 'db_dur_sec', 'n_epochs', 'hardware_version', 'drive_mode', 'country', 'city', 'front_footage_sec', 'rear_footage_sec', 'has_rear_camera', 'has_control', 'has_gps', 'has_imu', 'has_front_ts', 'has_rear_ts', 'has_mic_ts', 'has_speaker_ts', 'n_video_ts', 'has_video', 'complete', 'audio_only', 'shard', 'shard_member_bytes']

Primeras 5 filas:
                              folder  ride_id    device_ref_id  \
0  ride_100704_ethanc_20241213015117   100704  frodobot_ethanc   
1  ride_100736_ethanz_20241213101931   100736  frodobot_ethanz   
2  ride_100777_69bdbb_20241215062422   100777  frodobot_69bdbb   
3  ride_100778_69bdbb_20241215064245   100778  frodobot_69bdbb   
4  ride_100779_69bdbb_20241215070402   100779  frodobot_69bdbb

In [28]:
# Elegí el valor EXACTO tal como aparece arriba
PAIS = "Mexico"     # <-- EDITAR si el value_counts dice otra cosa

mx = df_meta[df_meta["country"] == PAIS]
print(f"{len(mx)} rides en {PAIS}, {mx['db_dur_sec'].sum()/3600:.1f} h totales\n")

flags = [c for c in ["has_control","has_gps","has_imu","has_front_ts",
                     "has_rear_camera","has_video","complete"] if c in mx.columns]
print("Cuantos rides cumplen cada condicion:")
print(mx[flags].sum().to_string())

# cuantos pasan TODOS los filtros que exige el select
base = mx[flags[:4]].all(axis=1) if len(flags) >= 4 else None
if base is not None:
    print(f"\nPasan control+gps+imu+front_ts: {base.sum()}")
    if "has_rear_camera" in mx.columns:
        print(f"  ...y ademas tienen camara trasera: {(base & mx['has_rear_camera']).sum()}")


189 rides en Mexico, 34.0 h totales

Cuantos rides cumplen cada condicion:
has_control        189
has_gps            189
has_imu            189
has_front_ts       189
has_rear_camera    189
has_video          189
complete           189

Pasan control+gps+imu+front_ts: 189
  ...y ademas tienen camara trasera: 189


In [29]:
# Inventario completo a Excel, por si lo querés mirar a mano
xlsx = WORK / "metadata_mexico.xlsx"
with __import__("pandas").ExcelWriter(xlsx, engine="openpyxl") as xl:
    mx.head(1_000_000).to_excel(xl, sheet_name="rides_mexico", index=False)
    df_meta["country"].value_counts().rename("n_rides").to_frame().to_excel(xl, sheet_name="por_pais")
print("->", xlsx)


-> /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/metadata_mexico.xlsx


## 3 — `select`: filtrar rides de México (no baja video)

Si el resultado te da pocos rides, sacá `--require-rear` y volvé a correr.

In [30]:
REQUIRE_REAR = True    # <-- poné False si el paso 2 mostro pocos rides con camara trasera

args = ["--countries", PAIS, "--out", SELECTED]
if REQUIRE_REAR:
    args.append("--require-rear")

run("1_descarga_filtrado.py", "select", *args)


$ /home/pablolube/IROS26-LaRovernetta/ML_model/.venv-data/bin/python 1_descarga_filtrado.py select --countries Mexico --out /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/selected_mexico.csv --require-rear    (cwd=/home/pablolube/IROS26-LaRovernetta/ML_model/scripts/pipe_videos_online)

[select] 189 rides seleccionados, 34.0 h totales -> /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/selected_mexico.csv
[select] rides por país:
country
Mexico    189

[exit 0]


0

In [31]:
import pandas as pd

sel = pd.read_csv(SELECTED)
print(f"{len(sel)} rides, {sel['db_dur_sec'].sum()/3600:.1f} h")
sel.to_excel(SELECTED.with_suffix(".xlsx"), index=False)
print("Excel ->", SELECTED.with_suffix(".xlsx"))
sel.head(20)


189 rides, 34.0 h
Excel -> /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/selected_mexico.xlsx


,folder,ride_id,device_ref_id,start_utc,source,db_dur_sec,n_epochs,hardware_version,drive_mode,country,...,has_front_ts,has_rear_ts,has_mic_ts,has_speaker_ts,n_video_ts,has_video,complete,audio_only,shard,shard_member_bytes
0,ride_120643_cpgq1m_20250523165042,120643,frodobot_cpgq1m,20250523165042,other,3857,1,6.1.31,mission,Mexico,...,1,1,0,1,743,1,1,0,data/minirover-0173.tar,859651395
1,ride_120616_cpgq1m_20250523130642,120616,frodobot_cpgq1m,20250523130642,other,3798,1,6.1.31,mission,Mexico,...,1,1,0,1,744,1,1,0,data/minirover-0171.tar,729046593
2,ride_120640_cpgq1m_20250523154815,120640,frodobot_cpgq1m,20250523154815,other,3705,1,6.1.31,mission,Mexico,...,1,1,0,1,745,1,1,0,data/minirover-0172.tar,993790913
3,ride_127333_cpgq1m_20250805153511,127333,frodobot_cpgq1m,20250805153511,other,3550,1,6.1.31,mission,Mexico,...,1,1,0,1,184,1,1,0,data/minirover-0266.tar,430122767
4,ride_120645_cpgq1m_20250523175532,120645,frodobot_cpgq1m,20250523175532,other,3018,1,6.1.31,mission,Mexico,...,1,1,0,1,625,1,1,0,data/minirover-0173.tar,874213246
5,ride_120514_cpgq1m_20250522162102,120514,frodobot_cpgq1m,20250522162102,other,2972,1,6.1.31,mission,Mexico,...,1,1,1,1,740,1,1,0,data/minirover-0169.tar,986769444
6,ride_127351_cpgq1m_20250805164223,127351,frodobot_cpgq1m,20250805164223,other,2551,1,6.1.31,mission,Mexico,...,1,1,0,1,184,1,1,0,data/minirover-0267.tar,460408590
7,ride_120531_cpgq1m_20250522190440,120531,frodobot_cpgq1m,20250522190440,other,2099,1,6.1.31,mission,Mexico,...,1,1,0,1,434,1,1,0,data/minirover-0171.tar,542403532
8,ride_121979_cpgq1m_20250610141809,121979,frodobot_cpgq1m,20250610141809,other,1574,1,6.1.31,mission,Mexico,...,1,1,0,1,186,1,1,0,data/minirover-0185.tar,464182228
9,ride_111444_cpgq1m_20250331131226,111444,frodobot_cpgq1m,20250331131226,other,1388,1,6.1.31,mission,Mexico,...,1,1,0,1,187,1,1,0,data/minirover-0087.tar,156961854


## 4 — `fetch`: bajar solo esos rides

**Este paso sí baja datos pesados.** Baja los shards que contienen los rides elegidos
y extrae solo esos. Empezá con pocos rides para medir cuánto tarda y cuánto ocupa.

In [32]:
# Opcional: recortar a los N rides mas largos para una primera prueba
N_PRUEBA = 10          # <-- None para bajar todos los seleccionados

rides_file = SELECTED
if N_PRUEBA:
    rides_file = WORK / f"selected_mexico_top{N_PRUEBA}.csv"
    sel.head(N_PRUEBA).to_csv(rides_file, index=False)
    print(f"Usando los {N_PRUEBA} rides mas largos -> {rides_file}")

run("1_descarga_filtrado.py", "fetch", "--rides", rides_file, "--raw-dir", RAW_DIR)


Usando los 10 rides mas largos -> /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/selected_mexico_top10.csv
$ /home/pablolube/IROS26-LaRovernetta/ML_model/.venv-data/bin/python 1_descarga_filtrado.py fetch --rides /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/selected_mexico_top10.csv --raw-dir /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/raw    (cwd=/home/pablolube/IROS26-LaRovernetta/ML_model/scripts/pipe_videos_online)

[fetch] 10 rides en 8 shards distintos
[fetch] (1/8) data/minirover-0087.tar — 1 ride(s)
[fetch] (2/8) data/minirover-0169.tar — 1 ride(s)
[fetch] (3/8) data/minirover-0171.tar — 2 ride(s)
[fetch] (4/8) data/minirover-0172.tar — 1 ride(s)
[fetch] (5/8) data/minirover-0173.tar — 2 ride(s)
[fetch] (6/8) data/minirover-0185.tar — 1 ride(s)
[fetch] (7/8) data/minirover-0266.tar — 1 ride(s)
[fetch] (8/8) data/minirover-0267.tar — 1 ride(s)
[fetch] listo -> /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/raw

[exit 0]


0

In [33]:
# Cuanto ocupo lo que bajaste
total = sum(p.stat().st_size for p in RAW_DIR.rglob("*") if p.is_file())
print(f"{RAW_DIR}: {total/1e9:.2f} GB en {len(list(RAW_DIR.iterdir()))} carpetas de ride")


/home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/raw: 6.50 GB en 10 carpetas de ride


## 5 — `clean`: sincronizar frame + GPS + control + IMU

Tolerancia de 500 ms, descarta GPS sin fix. Deja un `.synced.parquet` por ride
y un índice CSV.

In [40]:
run("1_descarga_filtrado.py", "clean",
    "--raw-dir", RAW_DIR, "--clean-dir", CLEAN_DIR, "--index", INDEX)


$ /home/pablolube/IROS26-LaRovernetta/ML_model/.venv-data/bin/python 1_descarga_filtrado.py clean --raw-dir /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/raw --clean-dir /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/clean --index /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/cleaned_index.csv    (cwd=/home/pablolube/IROS26-LaRovernetta/ML_model/scripts/pipe_videos_online)

[clean] 10 rides extraídos a limpiar
[clean] listo — índice en /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/cleaned_index.csv
[clean] % gps válido promedio: 100.0%

[exit 0]


0

In [41]:
import pandas as pd
idx = pd.read_csv(INDEX)
print(idx.describe(include="all").T.to_string())
idx.head(20)


                count unique                                                           top freq    mean         std  min      25%     50%     75%    max
ride_folder        10     10                             ride_111444_cpgq1m_20250331131226    1     NaN         NaN  NaN      NaN     NaN     NaN    NaN
n_frames_synced  10.0    NaN                                                           NaN  NaN    88.2   61.098827  3.0    36.75    87.0   126.5  198.0
frac_valid_gps   10.0    NaN                                                           NaN  NaN     1.0         0.0  1.0      1.0     1.0     1.0    1.0
dist_m           10.0    NaN                                                           NaN  NaN  300.48  212.948036  9.1  121.175  296.25  425.45  688.3
n_unique_cells   10.0    NaN                                                           NaN  NaN    44.1   34.996667  2.0    18.25    36.5    71.5  101.0
cells_json         10     10  ["2025-03-31_435705_-2072926", "2025-03-31_435705_-2

,ride_folder,n_frames_synced,frac_valid_gps,dist_m,n_unique_cells,cells_json
0,ride_111444_cpgq1m_20250331131226,3,1.0,9.1,2,"[""2025-03-31_435705_-2072926"", ""2025-03-31_435..."
1,ride_120645_cpgq1m_20250523175532,106,1.0,369.0,46,"[""2025-05-23_435712_-2072916"", ""2025-05-23_435..."
2,ride_121979_cpgq1m_20250610141809,198,1.0,688.3,101,"[""2025-06-10_435711_-2072918"", ""2025-06-10_435..."
3,ride_127351_cpgq1m_20250805164223,144,1.0,500.8,90,"[""2025-08-05_435711_-2072918"", ""2025-08-05_435..."
4,ride_120531_cpgq1m_20250522190440,60,1.0,198.8,44,"[""2025-05-22_435711_-2072918"", ""2025-05-22_435..."
5,ride_127333_cpgq1m_20250805153511,130,1.0,437.5,80,"[""2025-08-05_435711_-2072918"", ""2025-08-05_435..."
6,ride_120640_cpgq1m_20250523154815,68,1.0,223.5,22,"[""2025-05-23_435711_-2072919"", ""2025-05-23_435..."
7,ride_120616_cpgq1m_20250523130642,28,1.0,93.2,10,"[""2025-05-23_435711_-2072918"", ""2025-05-23_435..."
8,ride_120514_cpgq1m_20250522162102,116,1.0,389.3,29,"[""2025-05-22_435711_-2072918"", ""2025-05-22_435..."
9,ride_120643_cpgq1m_20250523165042,29,1.0,95.3,17,"[""2025-05-23_435704_-2072927"", ""2025-05-23_435..."


In [42]:
idx = pd.read_csv(INDEX)
rides_usados = pd.read_csv(rides_file)   # el top10.csv que usaste en fetch
merged = idx.merge(rides_usados[["folder", "db_dur_sec"]],
                    left_on="ride_folder", right_on="folder", how="left")
horas = merged["db_dur_sec"].sum() / 3600
print(f"{len(merged)} rides limpiados, {horas:.2f} h totales antes de curar")

10 rides limpiados, 7.92 h totales antes de curar


## 6 — `curate`: deduplicar por celda GPS

Evita quedarte con 30 rides de la misma cuadra. Para un primer fine-tune,
unas pocas horas bien diversas rinden más que muchas redundantes.

In [43]:
HORAS = 20      # <-- objetivo de horas; bajalo para el primer fine-tune

run("1_descarga_filtrado.py", "curate",
    "--index", INDEX, "--hours", HORAS, "--out", CURATED)


$ /home/pablolube/IROS26-LaRovernetta/ML_model/.venv-data/bin/python 1_descarga_filtrado.py curate --index /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/cleaned_index.csv --hours 20 --out /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/curated_mexico.csv    (cwd=/home/pablolube/IROS26-LaRovernetta/ML_model/scripts/pipe_videos_online)

[curate] 10 rides elegidos, 7.9 h, 358 celdas únicas -> /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/curated_mexico.csv

[exit 0]


0

In [44]:
cur = pd.read_csv(CURATED)
cur.to_excel(CURATED.with_suffix(".xlsx"), index=False)
print(f"{len(cur)} rides curados -> {CURATED.with_suffix('.xlsx')}")
cur.head(20)


10 rides curados -> /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/curated_mexico.xlsx


,ride_folder,n_frames_synced,frac_valid_gps,dist_m,n_unique_cells,db_dur_sec
0,ride_121979_cpgq1m_20250610141809,198,1.0,688.3,101,1574
1,ride_127351_cpgq1m_20250805164223,144,1.0,500.8,90,2551
2,ride_120645_cpgq1m_20250523175532,106,1.0,369.0,46,3018
3,ride_120531_cpgq1m_20250522190440,60,1.0,198.8,44,2099
4,ride_127333_cpgq1m_20250805153511,130,1.0,437.5,80,3550
5,ride_120640_cpgq1m_20250523154815,68,1.0,223.5,22,3705
6,ride_120514_cpgq1m_20250522162102,116,1.0,389.3,29,2972
7,ride_120643_cpgq1m_20250523165042,29,1.0,95.3,17,3857
8,ride_111444_cpgq1m_20250331131226,3,1.0,9.1,2,1388
9,ride_120616_cpgq1m_20250523130642,28,1.0,93.2,10,3798


## 7 — Chequeo antes de etiquetar

Lo que sigue **no está en este notebook** porque necesita GPU y trabajo manual:

1. Extraer frames de los rides curados.
2. `pipe_nuestras_rides/2_evaluar_con_modelo.py` sobre esos frames
   (`--guardar-mascaras` para pre-anotar y solo corregir).
3. Corregir máscaras (CVAT / Label Studio / `2b_corregir_mascaras.py`).
4. `3_armar_dataset.py --labeled ... --out training/FOLD_custom --val-frac 0.15`.
5. Entrenar en el clone de `facebookresearch/sam2` con `sam2.1_custom2.yaml`,
   **partiendo de `checkpoint_finetuned_v2.pt`**, nunca de cero.

Antes de eso, dos cosas a resolver:

- **GPU.** El config de referencia es resolución 1024, batch 8. Una 2080 de 8 GB
  no lo aguanta: hay que bajar batch o buscar cluster.
- **IoU.** Hoy la validación es a ojo. Con datos de un país nuevo, el riesgo es
  olvido catastrófico, y sin métrica no lo ves. Está marcado como pendiente en
  `1_Doc tecnica y comandos.md`.


In [39]:
print("Resumen de la corrida\n" + "="*40)
for nombre, ruta in [("metadata", META_DIR), ("selected", SELECTED), ("raw", RAW_DIR),
                     ("clean", CLEAN_DIR), ("index", INDEX), ("curated", CURATED)]:
    p = Path(ruta)
    estado = "OK" if p.exists() else "falta"
    print(f"  {nombre:9s} {estado:5s}  {p}")


Resumen de la corrida
  metadata  OK     /home/pablolube/IROS26-LaRovernetta/ML_model/frodobots_metadata
  selected  OK     /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/selected_mexico.csv
  raw       OK     /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/raw
  clean     OK     /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/clean
  index     OK     /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/cleaned_index.csv
  curated   OK     /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/curated_mexico.csv
